# Curanex: kidney stone on CT - threshold set on CT-KIDNEY, tested on a third collection (CPU)

The axial-trained kidney stone model ranked CT-KIDNEY well (AUC 0.921) but its threshold caught 59%. **Rules fixed before running
(2026-11-12):** model unchanged; stone score = P(stone); threshold = highest with sensitivity >= 0.85 on CT-KIDNEY stone vs normal
(already used once, now the calibration set; copies of training images removed as before). **Outside test, scored once:** the
Roboflow kidney stone collection (safurahajiheidari, all splits): stone = image has a stone box, no stone = empty or missing label;
images within 6 bits (dHash) of any training or calibration image are removed. Verified if sensitivity and AUC are both >= 0.80.

In [ ]:
!pip install -q onnxruntime

In [ ]:
import glob, json, os
import numpy as np, onnxruntime as ort
from PIL import Image
from sklearn.metrics import roc_auc_score
meta = json.load(open(glob.glob("/kaggle/input/**/kidney_stone_axial.json", recursive=True)[0]))
sess = ort.InferenceSession(glob.glob("/kaggle/input/**/kidney_stone_axial.onnx", recursive=True)[0])
MEAN, STD = np.array(meta["mean"], np.float32), np.array(meta["std"], np.float32)


def dhash(p):
    a = np.asarray(Image.open(p).convert("L").resize((9, 8)), dtype=np.float32)
    return int("".join("1" if b else "0" for b in (a[:, 1:] > a[:, :-1]).ravel()), 2)


def near(h, bank):
    return bool((np.unpackbits(np.bitwise_xor(bank, np.uint64(h)).view(np.uint8).reshape(-1, 8), axis=1).sum(1) <= 6).any())


def prep(f):
    im = Image.open(f).convert("RGB"); w, h = im.size; s = 256 / min(w, h)
    im = im.resize((max(224, round(w * s)), max(224, round(h * s))), Image.BILINEAR); w, h = im.size
    l, t = (w - 224) // 2, (h - 224) // 2
    return ((np.asarray(im.crop((l, t, l + 224, t + 224)), np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1)


def scores(files):
    out = []
    for i in range(0, len(files), 64):
        out += list(sess.run(None, {"image": np.stack([prep(f) for f in files[i:i + 64]])})[0][:, 0])
    return np.array(out)


def report(y, p, t):
    y, p = np.array(y), np.array(p); f = p >= t
    r = {"n": int(len(y)), "positives": int(y.sum()), "sensitivity": round(float(f[y == 1].mean()), 3),
         "specificity": round(float((~f[y == 0]).mean()), 3), "auc": round(float(roc_auc_score(y, p)), 3)}
    low = min(r["sensitivity"], r["auc"]); r["status"] = "verified" if low >= 0.80 else "live" if low >= 0.75 else "off"
    return r


img = lambda f: f.lower().endswith((".jpg", ".jpeg", ".png"))
train = [f for f in glob.glob("/kaggle/input/**/Original Dataset/**/*", recursive=True) if img(f)]
bank = np.array([dhash(f) for f in train], dtype=np.uint64)
cal = []
for f in glob.glob("/kaggle/input/**/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/**/*", recursive=True):
    folder = os.path.basename(os.path.dirname(f)).lower()
    if img(f) and folder in ("stone", "normal") and not near(dhash(f), bank):
        cal.append((f, int(folder == "stone")))
bank2 = np.concatenate([bank, np.array([dhash(f) for f, _ in cal], dtype=np.uint64)])
test, dropped = [], 0
for f in glob.glob("/kaggle/input/**/images/*", recursive=True):
    if not img(f) or "CT-KIDNEY" in f or "Original Dataset" in f:
        continue
    if near(dhash(f), bank2):
        dropped += 1; continue
    lab = os.path.join(os.path.dirname(os.path.dirname(f)), "labels", os.path.splitext(os.path.basename(f))[0] + ".txt")
    test.append((f, int(os.path.exists(lab) and open(lab).read().strip() != "")))
print("calibration (CT-KIDNEY):", len(cal), sum(y for _, y in cal), "| test:", len(test), sum(y for _, y in test), "copies removed:", dropped)
assert len(test) >= 50 and 0 < sum(y for _, y in test) < len(test)
pc = scores([f for f, _ in cal]); yc = np.array([y for _, y in cal])
T = float(max(v for v in np.unique(np.round(pc, 6)) if (pc[yc == 1] >= v).mean() >= 0.85))
print("threshold", T, "calibration:", report(yc, pc, T))
res = report([y for _, y in test], scores([f for f, _ in test]), T)
print("OUTSIDE TEST on Roboflow kidney stone set (once):", res)
json.dump({"threshold": T, "outside": res}, open("/kaggle/working/kidney_third.json", "w"), indent=1)